In [2]:
# ==========================================================
# DAY 16 - MASTER DATASET CREATION
# ==========================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

INPUT_FILE = PROJECT_ROOT / "03_processed_data/vendor_performance.csv"
OUTPUT_FILE = PROJECT_ROOT / "03_processed_data/master_vendor_dataset.csv"

df = pd.read_csv(INPUT_FILE)

print("Loaded:", df.shape)

# Rename for consistency
df.columns = [
    "vendor_name",
    "total_quantity",
    "avg_cost",
    "avg_defect",
    "total_cost"
]

# Create efficiency metric
df["efficiency"] = df["total_quantity"] / df["total_cost"]

# Normalize metrics (for scoring)
df["cost_score"] = (df["avg_cost"] - df["avg_cost"].min()) / (df["avg_cost"].max() - df["avg_cost"].min())
df["defect_score"] = (df["avg_defect"] - df["avg_defect"].min()) / (df["avg_defect"].max() - df["avg_defect"].min())

# Lower is better → invert
df["final_score"] = (1 - df["cost_score"]) * 0.5 + (1 - df["defect_score"]) * 0.5

# ----------------------------------------------------------
# ADDITIONAL METRICS
# ----------------------------------------------------------

# Cost per unit normalization
df["cost_per_unit"] = df["total_cost"] / df["total_quantity"]

# Defect impact cost (estimated loss)
df["defect_impact"] = df["total_quantity"] * (df["avg_defect"] / 100) * df["avg_cost"]

# Value score (profit-like proxy)
df["value_score"] = df["total_quantity"] * (1 - df["avg_defect"] / 100)

# Z-score normalization (advanced)
df["cost_zscore"] = (df["avg_cost"] - df["avg_cost"].mean()) / df["avg_cost"].std()
df["defect_zscore"] = (df["avg_defect"] - df["avg_defect"].mean()) / df["avg_defect"].std()

# Save updated dataset
df.to_csv(OUTPUT_FILE, index=False)


Loaded: (20, 5)
